# Tier 2: 750-Step Proof (TPU v5e Optimized)

**Purpose**: Catch late bloomers. Verify Tier 1 winners aren't flukes.

| Decision | Condition |
|----------|----------|
| ❌ Kill | step-750 BPB **> 1.75** |
| ⚠️ Marginal | step-750 BPB **1.70 – 1.75** |
| ✅ Promote to Tier 3 | step-750 BPB **< 1.70** |

**Baseline reference**: 1.7214 BPB @ step 750 (T4, 32k batch, seed 42)

### Changes in this version:
1. **Automated Verdict**: Cell 4 now parses `final_summary.json` automatically.
2. **XLA Optimizations**: Includes `NO_COMPILE=1` and `device_sync()` awareness.
3. **TPU Detection**: Ensures `EVAL_CACHE=0` to prevent host CPU stalls.

In [ ]:
# [CELL 1] CLONE
%cd /content
!rm -rf uniform-int4
!git clone https://github.com/jmoncayo-pursuit/parameter-golf-uniform-int4.git uniform-int4
print('✅ Repo cloned')

In [ ]:
# [CELL 2] ASSETS
%cd /content/uniform-int4
!mkdir -p data/tokenizers data/datasets/fineweb10B_sp1024
!wget -q -O data/tokenizers/fineweb_1024_bpe.model \
  https://huggingface.co/datasets/willdepueoai/parameter-golf/resolve/main/datasets/tokenizers/fineweb_1024_bpe.model
!wget -q -O data/datasets/fineweb10B_sp1024/fineweb_val_000000.bin \
  https://huggingface.co/datasets/willdepueoai/parameter-golf/resolve/main/datasets/datasets/fineweb10B_sp1024/fineweb_val_000000.bin
!wget -q -O data/datasets/fineweb10B_sp1024/fineweb_train_000001.bin \
  https://huggingface.co/datasets/willdepueoai/parameter-golf/resolve/main/datasets/datasets/fineweb10B_sp1024/fineweb_train_000001.bin
!pip install -q sentencepiece zstandard PyYAML huggingface-hub datasets tqdm

import os
assert os.path.exists('data/tokenizers/fineweb_1024_bpe.model')
assert os.path.exists('data/datasets/fineweb10B_sp1024/fineweb_val_000000.bin')
print('✅ All assets verified')

In [ ]:
# [CELL 3] 750-STEP PROOF
%cd /content/uniform-int4
# Ensure EVAL_CACHE=0 to avoid TPU host CPU stalls
!env NO_COMPILE=1 ITERATIONS=750 VAL_LOSS_EVERY=250 EVAL_CACHE=0 python3 train_gpt.py

In [ ]:
# [CELL 4] VERDICT (AUTOMATED)
import json
import os

if os.path.exists('final_summary.json'):
    with open('final_summary.json', 'r') as f:
        summary = json.load(f)
    
    bpb = summary.get('roundtrip_val_bpb', summary.get('val_bpb'))
    BASELINE = 1.7214
    
    print(f'\n--- FINAL VERDICT ---')
    print(f'Roundtrip BPB: {bpb:.4f}')
    print(f'Artifact Size: {summary.get("total_submission_size", 0) / 1024 / 1024:.2f} MB')
    
    if bpb > 1.75:
        print(f'❌ KILL — {bpb:.4f} won\'t reach leaderboard.')
    elif bpb > BASELINE:
        print(f'⚠️ MARGINAL — {bpb:.4f} is close to baseline ({BASELINE}).')
    elif bpb > 1.70:
        print(f'🔶 STRONG — {bpb:.4f} beats baseline ({BASELINE}). Promote to Tier 3.')
    else:
        print(f'✅ PROMOTE — {bpb:.4f} significantly beats baseline ({BASELINE}). Run Tier 3 full proof.')
else:
    print('❌ ERROR: final_summary.json not found. Did the training crash?')
